<a href="https://colab.research.google.com/github/AgadaOlshtein/Automation-Course--PLC-project/blob/main/Copy_of_Bogusness_Detection_using_LLMs_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center><br><font size=6><b><font color='#7849FF'>DataNights</font></b></font><br>
    <br><font size=4><b>Bogusness Detection using LLMs</b></u></font><br>
    <br><font color ='gray'><b>18/10/2025</b></i><br></center>

<img src="https://aigptjournal.com/wp-content/uploads/2024/06/2b1f51e7-930b-45c5-8409-78ee95be0816.gif" alt="Alt text that describes the graphic" title="Title text" />

<font size =5 ><u><b>Table of Contents:</font></u></b>
* [🎯 Goal of the Exercise](#zero-bullet)
* [1️⃣ Setup and Data Loading](#one-bullet)
* [2️⃣ Initialize the OpenAI Client](#two-bullet)
* [3️⃣ Define the LLM Bogusness Prediction Function](#three-bullet)
* [4️⃣ Run the Prediction on All Data](#four-bullet)
* [5️⃣ Model Evaluation](#five-bullet)



<a class="anchor" id="zero-bullet"></a>

# 🎯 Goal of the Exercise

<div class="alert alert-block alert-success">
<font size=4 color='#7849FF'>
<b><center>Bogusness Detection using LLMs</center></b>
</font>

<p>
In this exercise, we explore how Large Language Models (LLMs) can be leveraged to detect <b>bogus or fraudulent data entries</b> in tabular datasets.
Bogus data may appear as gibberish text, incomplete fields, placeholder emails, suspicious phone numbers, or other inconsistencies that indicate the data is likely invalid or artificially generated.
</p>

<p>
The goal is to train students to:
<ul>
    <li>Understand how to design effective prompts for LLMs.</li>
    <li>Generate predictions of <b>bogusness scores</b> and accompanying reasons for each data entry.</li>
    <li>Parse LLM outputs in a structured format (JSON) for further analysis.</li>
    <li>Compare LLM predictions against a known <b>ground truth label</b> (fraud / not fraud).</li>
    <li>Visualize and evaluate model performance using metrics such as <b>precision</b> and <b>recall</b>, and explore threshold tuning.</li>
</ul>
</p>

<p>
This hands-on exercise demonstrates:
<ul>
    <li>How to interact programmatically with LLM APIs (OpenAI GPT models such as <code>gpt-4o-mini</code> and <code>gpt-4-turbo</code>).</li>
    <li>Techniques for prompt engineering to improve model reasoning and output reliability.</li>
    <li>Practical evaluation of predictions, including box plots and precision-recall analysis.</li>
</ul>
</p>

<p>
By the end of this exercise, students will gain experience in combining <b>prompt engineering</b>, <b>LLM inference</b>, and <b>data evaluation metrics</b> to detect anomalies and bogus entries in realistic datasets, a crucial skill in <b>fraud detection</b> and <b>data integrity</b> tasks.
</p>
</div>


<div class="alert alert-block alert-info">
<font size=3 color='#000000'>
<b>Each Student Creates Their Own API Key</b><br><br>
Ask students to sign up at <a href="https://platform.openai.com">https://platform.openai.com</a>.<br><br>
Go to View API Keys → Create new secret key.<br><br>
In the notebook, they’ll set:<br>
<pre>
import os
os.environ["OPENAI_API_KEY"] = "sk-xxxx..."
</pre>
</font>
</div>


<a class="anchor" id="one-bullet"></a>

In [1]:
%pip install names openai pandas matplotlib seaborn scikit-learn tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 789.1/789.1 kB 11.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for names: filename=names-0.3.0-py3-none-any.whl size=803681 sha256=dd539b87de99f156a56a5e648fb5a4f2ef1a6375a479f7d704ad8ae7f5d43ed4
  Stored in directory: /root/.cache/pip/wheels/c7/f0/8f/de9f15941cd988c39b82703fa04cb2d550ba5867f13c6da052
Successfully built names


# 1️⃣ Setup and Data Loading

In [2]:
import pandas as pd
import random
import names  # pip install names

# ------------------------
# Create a diverse pool
# ------------------------

# Realistic names using `names` package
good_names = [f"{names.get_first_name()} {names.get_last_name()}" for _ in range(200)]

# Some additional realistic company-like names
good_names += ["Alpha Tech", "Beta Solutions", "Gamma Corp", "Delta LLC", "Epsilon Inc"]

# Bogus/gibberish names
bad_names = ["Hfgj Yxkl", "Xyz Corp", "asdf qwer", "", None, "qwertyuiop", "lorem ipsum", "zzzzzz", "aaaaa", "nnnnnn"]

# Emails
good_emails = [f"{n.split()[0].lower()}@example.com" for n in good_names if n]
good_emails += ["contact@company.com", "info@business.org", "support@service.net"]
bad_emails = ["lnfjdndfljdnslj@gmail.com", "aaa@random.net", "", None, "fake_email@", "user@unknown."]

# Phones
good_phones = [f"{random.randint(200,999)}{random.randint(1000000,9999999)}" for _ in range(200)]
bad_phones = ["0000000000", "1111111111", "2223334444", "", None, "12345", "9999999"]

# Addresses
good_addresses = [
    "123 Main St", "456 Oak Ave", "789 Pine Rd", "12 Elm St", "321 Maple Dr",
    "654 Cedar Blvd", "88 Birch Ln", "77 Walnut St", "55 Cherry Ave", "99 Poplar Rd"
]
bad_addresses = ["999 Random Blvd", "Unknown", "No Address", "", None, "XXXXX", "12345"]

# ------------------------
# Generate dataset
# ------------------------
rows = []

for i in range(50):
    if random.random() < 0.7:  # 70% good
        name = random.choice(good_names)
        email = random.choice(good_emails)
        phone = random.choice(good_phones)
        address = random.choice(good_addresses)
        label = 0
    else:  # 30% bad
        name = random.choice(bad_names)
        email = random.choice(bad_emails)
        phone = random.choice(bad_phones)
        address = random.choice(bad_addresses)
        label = 1
    rows.append([name, email, phone, address, label])

# Create DataFrame
df_sample = pd.DataFrame(rows, columns=["name", "email", "phone", "address", "label"])

# Save CSV
df_sample.to_csv("sample_bogus_data_500_diverse.csv", index=False)

print("sample_bogus_data_500_diverse.csv created!")
df_sample.head(10)

sample_bogus_data_500_diverse.csv created!


,name,email,phone,address,label
0,Bertie Burnham,tosha@example.com,5121529944,654 Cedar Blvd,0
1,Eleanor Giddins,lillian@example.com,7991735065,88 Birch Ln,0
2,Anna House,jessica@example.com,3008995978,12 Elm St,0
3,Dorothy Gallemore,amy@example.com,7804676181,77 Walnut St,0
4,lorem ipsum,lnfjdndfljdnslj@gmail.com,2223334444,999 Random Blvd,1
5,Hfgj Yxkl,aaa@random.net,1111111111,Unknown,1
6,Ulysses Lairy,anna@example.com,8751081968,12 Elm St,0
7,Lionel Bryce,joseph@example.com,9683311936,77 Walnut St,0
8,Patrica Johnson,lisa@example.com,2302267870,12 Elm St,0
9,Eleanor Giddins,brian@example.com,5976765474,77 Walnut St,0


<a class="anchor" id="two-bullet"></a>

# 2️⃣ Initialize the OpenAI Client

In [13]:
from openai import OpenAI
import os

from google.colab import userdata

assert userdata.get('OPENAI_API_KEY'), "Please set your OpenAI API key first."
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
client = OpenAI()

# You can switch between models
MODEL = "gpt-4o-mini"  # or "gpt-4-turbo"

<a class="anchor" id="three-bullet"></a>

# 3️⃣ Define the LLM Bogusness Prediction Function

<div class="alert alert-block alert-info">
<b>💡<u>Goal</u>💡</b>
<ul>
    <li>Customize the prompt to improve the model’s bogusness detection.</li>
    <li>Locate the <code>predict_bogusness</code> function in the notebook.</li>
</ul>

<b>What you can edit:</b>
<ul>
    <li><b>Instruction text:</b> Make the task clearer or add specific constraints.</li>
    <li><b>Examples:</b> Include few-shot examples of input and expected output.</li>
    <li><b>Output format:</b> Force JSON, provide a list of reasons, or use another structured format.</li>
    <li><b>Level of detail:</b> Request short explanations or detailed reasoning.</li>
    <li><b>Tone or style:</b> For example, “Answer like a professional fraud analyst.”</li>
</ul>
</div>


In [5]:
def predict_bogusness(row):
    prompt = f"""
    You are a fraud detection assistant.
    Given the following data:
    - Name: {row['name']}
    - Email: {row['email']}
    - Address: {row['address']}
    - Phone: {row['phone']}

    Determine:
    1. A bogusness score (0–100)
    2. A short explanation for the score.

    Return the result as JSON with the fields: 'bogus_score' and 'reason'.
    """

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
    )

    import json
    import re
    try:
        unparsed_response = response.choices[0].message.content
        # parse the result into a json by first removing the prefix ```json and suffix ```
        parsed_response = re.sub(r"^```json|```$", "", unparsed_response.strip()).strip()
        result = json.loads(parsed_response)
    except:
        result = {"bogus_score": None, "reason": response.choices[0].message.content}
    return result

<a class="anchor" id="four-bullet"></a>

# 4️⃣ Run the Prediction on All Data

In [15]:
from tqdm import tqdm

results = []
for _, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
    res = predict_bogusness(row)
    results.append(res)

df_results = pd.concat([df_sample, pd.DataFrame(results)], axis=1)
df_results.to_csv("bogusness_results.csv", index=False)
df_results.head()

  0%|          | 0/50 [00:03<?, ?it/s]


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

<a class="anchor" id="five-bullet"></a>

# 5️⃣ Model Evaluation

In [9]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_score, recall_score

# 1️⃣ Function to compute binary predictions based on threshold
def apply_threshold(df, score_column='bogus_score', threshold=50):
    """
    Convert continuous bogus scores (0-100) into binary predictions (0/1)
    using the specified threshold.
    """
    df['predicted_label'] = (df[score_column] >= threshold).astype(int)
    return df

# 2️⃣ Apply threshold
THRESHOLD = 10  # You can experiment with different thresholds
df_results = apply_threshold(df_results, score_column='bogus_score', threshold=THRESHOLD)

# 3️⃣ Compute precision and recall
precision = precision_score(df_results['label'], df_results['predicted_label'])
recall = recall_score(df_results['label'], df_results['predicted_label'])

print(f"Threshold: {THRESHOLD}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")

# 4️⃣ Box plot: y-axis = bogus prediction score, x-axis = true label (fraud/not fraud)
plt.figure(figsize=(8,6))
sns.boxplot(x='label', y='bogus_score', data=df_results)
plt.xlabel("Fraud Label (0=Not Fraud, 1=Fraud)")
plt.ylabel("Bogus Prediction Score")
plt.title("Distribution of Bogus Scores by True Label")
plt.show()

NameError: name 'df_results' is not defined